# Battle 通用远程 GPU Worker（单 cell 版）

**用法：填好下方唯一代码 cell 顶部的 `HUB_URL` / `HUB_TOKEN`（push 模式再加 `PUSH_TOKEN`），Run 它，其余全自动。**

| 自动步骤 | 说明 |
|---|---|
| 设备探测 | CUDA（多卡开关 `USE_MULTI_GPU`）/ TPU（内核不碰设备，独占约束）/ CPU |
| 保活 | Colab 每 60s 模拟点击 · Kaggle 每 120s 心跳 |
| 代码引导 | 从 hub `GET /code` 拉 code.zip；hub 还没发布过 job 时**每 30s 等待重试**（先起 worker 后起 trainer 是合法顺序）；此后代码变更随 job 热替换 |
| 任务执行 | worker 监督循环：代码热替换（退 86 自动重启）· 崩溃退避自动重启（≤`MAX_WORKER_RESTARTS`）· 空闲满 `max_idle` 干净退出 |
| 任务类型 | PPO（kind=ppo）与 BC（kind=bc）同一 worker，manifest 自带全部上下文，无需区分 |

**连接信息**：`HUB_URL` = 控制台 cloudflared 卡片隧道 URL（= `rl.remote_hubs.<课程>`）；
`HUB_TOKEN` = 训练机 `nn-training/rl-config.json` 的 `rl.remote_token`。多课程并行每课一隧道，别拿别课的 URL。

**停止**：中断本 cell（■）= 干净停机（子进程全回收）。
**配额**：Kaggle 30h GPU/周、单次 9h（`MAX_SESSION_HOURS` 调小省配额）；Colab 免费 ~90min 空闲回收（保活顶着）。

### 常见问题

| 症状 | 处理 |
|---|---|
| `FATAL: hub /ping HTTP 401` | token 与 hub 端 `rl.remote_token` 不一致 |
| `FATAL: hub 不可达` | 隧道过期——控制台重启 cloudflared 后更新 URL |
| 等待 code.zip 超 1h | hub 侧从未发布过任何 job——确认训练机 trainer/BC 编排器已启动 |
| `TPU ... Device or resource busy` | TPU 独占设备被持有；本内核占用只能 Runtime → Restart session（cell 输出里有占用者诊断） |
| payload 校验失败 | 传输损坏，自动重试；hub 幂等保证不重复训练 |

历史版本（多 cell 详版）见 git 历史；push 模式 cloudflared 由 cell 自动安装。


In [ ]:
# @title 一键连接 hub —— 填参数 → Run 本 cell，其余全自动
# ============================================================================
# Battle 通用远程 GPU Worker（单 cell 版，2026-09-13 重构自多 cell 版）
#
# 用法：只改下面 ⚙️ 连接参数 三行（MODE / HUB_URL / HUB_TOKEN），然后运行本 cell。
# 之后全部自动：
#   [自动] 平台/设备探测（CUDA / TPU / CPU；TPU 时内核绝不触碰设备——独占约束）
#   [自动] 保活（Colab 每 60s 模拟点击 / Kaggle 每 120s 心跳打印）
#   [自动] 从 hub /code 拉执行代码（hub 还没发布过 job 时**等待重试**——训练机
#          第一次发布 job 就会生成 code.zip；此后代码变更随 job 热替换）
#   [自动] worker 监督循环：代码热替换（退出码 86 → 自动重启加载新代码）；
#          崩溃 → 指数退避自动重启；空闲满 max_idle 自动干净退出
#   [自动] push 模式：自动安装 cloudflared + 打印隧道 URL + 等待 hub 推送
# 停止：中断本 cell（■）= 干净停机（子进程全部回收，不留孤儿）。
# 任务类型：hub 下发的 PPO（kind=ppo）与 BC（kind=bc）job 由同一 worker 处理，
# 无需任何区分——job manifest 自带全部上下文（plan/bc-cloud-integration.plan.md）。
#
# 连接信息哪里拿：
#   HUB_URL   = 控制台 cloudflared 卡片的隧道 URL（即 rl-config rl.remote_hubs.<课程>）
#   HUB_TOKEN = rl-config 的 rl.remote_token（训练机 nn-training/rl-config.json）
#   多课程并行时每课一隧道——拿目标课程自己的 URL，别拿别课的（串线）。
# ============================================================================

import os
import subprocess
import sys
import time

# ──────────────────────────── ⚙️ 连接参数（通常只改这里） ────────────────────────────
MODE = "pull"            # "pull"（worker 轮询 hub，推荐）或 "push"（hub 推过来）

# Pull 模式：
HUB_URL = "https://your-tunnel.trycloudflare.com"
HUB_TOKEN = "YOUR_TOKEN_HERE"

# Push 模式（MODE="push" 时才需要）：
PUSH_PORT = 8790                    # GPU 侧服务端口（cloudflared 暴露它）
PUSH_TOKEN = "YOUR_TOKEN_HERE"      # 与 hub 侧 rl.remote_token 一致
CLOUDFLARED_PATH = ""               # 留空 = 自动查找/自动安装

# ──────────────────────────── ⚙️ 行为参数（一般不用动） ────────────────────────────
MAX_SESSION_HOURS = 9      # 会话上限：Kaggle GPU 单次最长 9h；短腿课程调小省配额
POLL_INTERVAL_SEC = 5      # pull 轮询周期
IDLE_FLOOR_SEC = 3600      # worker 空闲退出的下限（实际 max_idle = max(本值, (会话-1)h)）
MAX_WORKER_RESTARTS = 5    # worker 崩溃后的自动重启次数上限（退避 30s×次数）
USE_MULTI_GPU = False      # True = 多卡 DataParallel（改变梯度归约顺序——新实验臂开关）

PLATFORM = (
    "colab"
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
    else "kaggle"
)


def _log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)


# ════════════════════════ 1. 设备探测：CUDA → TPU → CPU ════════════════════════
# TPU 硬约束（2026-09-10/11 线上事故收口）：
#   1. 必须 torch_xla 且 torch / torch_xla 版本严格配对（镜像自带）——绝不 pip 覆盖；
#   2. PJRT_DEVICE=TPU 只影响 worker 子进程（本内核不读）；
#   3. ★ 本内核绝不 import torch_xla：/dev/vfio/*（v5e）与 /dev/accel*（v3/v4）是
#      独占 PCI 直通设备——内核碰过，worker 子进程必报
#      `TPU initialization failed: open(/dev/vfio/0): Device or resource busy`。
#      探测只查「装没装 / 谁占着」，设备一律留给子进程。
import glob


def _tpu_device_nodes() -> list:
    """本 runtime 映射进来的 TPU 设备节点（只 glob 目录名，绝不 open）。"""
    return sorted(glob.glob("/dev/vfio/*")) + sorted(glob.glob("/dev/accel*"))


def _tpu_holders() -> list:
    """谁正占着 TPU 设备节点（pid + cmdline）；扫 /proc/*/fd，同样不 open 设备。"""
    out = []
    for fdlink in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            tgt = os.readlink(fdlink)
        except OSError:
            continue
        if not tgt.startswith("/dev/vfio"):
            continue
        pid = int(fdlink.split("/")[2])
        try:
            with open(f"/proc/{pid}/cmdline", "rb") as f:
                cmd = f.read().replace(b"\0", b" ").decode("utf-8", "replace").strip()
        except OSError:
            cmd = "?"
        out.append((pid, tgt, cmd))
    return out


DEVICE = "cpu"
_n_gpu = 0
try:
    import torch

    if torch.cuda.is_available():
        DEVICE = "cuda"
        _n_gpu = torch.cuda.device_count()
        if USE_MULTI_GPU and _n_gpu > 1:
            DEVICE = "cuda-dp"
        _log(f"torch {torch.__version__}, CUDA 可见 {_n_gpu} 张 GPU")
        for _i in range(_n_gpu):
            _log(f"  [{_i}] {torch.cuda.get_device_name(_i)}")
        if DEVICE == "cuda-dp":
            _log(f"  -> DataParallel 跨 {_n_gpu} 卡（梯度归约顺序变化，与单卡 run 数值不可逐位比）")
        elif _n_gpu > 1:
            _log(f"  -> 只用第 0 张卡（要跨卡把 USE_MULTI_GPU 改 True）")
    else:
        import importlib.metadata as _md
        import importlib.util as _iu

        if _iu.find_spec("torch_xla") is not None:
            try:
                _xla_ver = _md.version("torch_xla")
            except Exception:
                _xla_ver = "（已安装，版本未知）"
            os.environ.setdefault("PJRT_DEVICE", "TPU")
            DEVICE = "tpu"
            _log(f"torch_xla {_xla_ver}（内核未导入——设备留给 worker 子进程）")
            _log(f"  TPU 设备节点: {_tpu_device_nodes() or '（没看到 /dev/vfio* 或 /dev/accel*）'}")
            _holders = _tpu_holders()
            for _pid, _tgt, _cmd in _holders:
                _tag = "★本内核★" if _pid == os.getpid() else "其它进程"
                _log(f"  ⚠ {_tgt} 已被占用: pid={_pid} [{_tag}] {_cmd[:90]}")
            if _holders:
                _log("  ⚠ 占用存在时 worker 子进程必报 busy；占用者是本内核 → 只能 Runtime → Restart session")
            else:
                _log("  vfio 无占用 —— 设备空闲，worker 子进程可正常领取")
        else:
            _log("无 CUDA / 无 TPU（torch_xla 未安装）→ CPU")
except ImportError:
    _log("torch 不可用 → CPU（job 执行会失败——Kaggle/Colab 请选 GPU/TPU 运行时）")
_log(f"DEVICE = {DEVICE}")

# ════════════════════════ 2. 保活线程 ════════════════════════
import threading

KEEPALIVE_STOP = threading.Event()

if PLATFORM == "colab":

    def _keepalive_loop():
        """每 60s 点一次 Colab 的 connect 按钮防超时。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            try:
                from IPython.display import Javascript, display

                display(
                    Javascript(
                        "function clickConnect(){document.querySelector("
                        '"colab-connect-button")?.click();}setTimeout(clickConnect,1000);'
                    )
                )
                n += 1
            except Exception:
                pass
            KEEPALIVE_STOP.wait(60)
        _log(f"[keepalive] stopped after {n} pings")

else:

    def _keepalive_loop():
        """每 120s 心跳打印，防 Kaggle 空闲回收。"""
        n = 0
        while not KEEPALIVE_STOP.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] alive ({n * 2} min)", flush=True)
            n += 1
            KEEPALIVE_STOP.wait(120)
        _log(f"[keepalive] stopped after {n} pings")


threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log(f"Keepalive 已启动（{PLATFORM}，每 {'60' if PLATFORM == 'colab' else '120'}s）")

# ════════════════════════ 3. hub 连接 + 代码引导（pull/push 共用） ════════════════════════
import hashlib
import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path


def _hub_get(path: str, token: str, timeout: float = 120) -> bytes:
    """hub GET（Bearer）；HTTPError 直接上抛（调用方按状态码分诊）。"""
    req = urllib.request.Request(
        f"{HUB_URL.rstrip('/')}{path}", headers={"Authorization": f"Bearer {token}"}
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def _fetch_code_zip(token: str, hub_label: str) -> bytes:
    """GET /code → code.zip 字节；**404 = 等待重试**（hub 尚无任何 job 发布）。

    顺序无关的自主接入的关键：训练机（run_rl / run_bc）第一次发布 job 时才把
    code.zip 写到 job_root——云端 worker 先起、trainer 后起完全合法，这里等它。
    401/403 = token 错 → 立即失败（重试无意义）。"""
    deadline = time.time() + 3600
    attempt = 0
    while True:
        attempt += 1
        try:
            raw = _hub_get("/code", token)
            _log(f"code.zip 就绪: {len(raw)} bytes, sha256={hashlib.sha256(raw).hexdigest()[:12]}…（第 {attempt} 次尝试）")
            return raw
        except urllib.error.HTTPError as e:
            if e.code in (401, 403):
                _log(f"FATAL: {hub_label} HTTP {e.code} — token 不匹配（检查 {'HUB_TOKEN' if hub_label == 'hub' else 'PUSH_TOKEN'}）")
                raise
            if e.code == 404:
                if time.time() > deadline:
                    _log("FATAL: 等待 code.zip 超 1h——hub 侧从未发布过任何 job，检查训练机是否已启动")
                    raise
                if attempt == 1 or attempt % 6 == 0:
                    _log(f"hub 尚无 code.zip（训练机还没发布过 job）——每 30s 重试，先起 worker 后起 trainer 是合法顺序…")
                time.sleep(30)
                continue
            _log(f"{hub_label} /code HTTP {e.code} —— 30s 后重试")
            time.sleep(30)
        except Exception as e:
            _log(f"{hub_label} 连接异常（{type(e).__name__}: {e}）—— 30s 后重试")
            time.sleep(30)


def _unpack_code(code_raw: bytes, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(code_raw)) as z:
        z.extractall(dest)
    _log(f"code.zip 解包 -> {dest}")
    return dest


# ════════════════════════ 4. Pull 模式 runner ════════════════════════
def run_pull_worker() -> int:
    """连接 hub 轮询领 job（PPO 与 BC 同 worker），返回退出码：
    0 = 干净退出（空闲满/会话到顶）；-2 = 配置致命（不重启）；其它非 0 = 可重启的失败。"""
    work_dir = Path("/tmp/remote-worker")
    work_dir.mkdir(parents=True, exist_ok=True)

    _log(f"连接 hub: {HUB_URL} （/ping 探测…）")
    try:
        _hub_get("/ping", HUB_TOKEN, timeout=15)
        _log("hub ping OK")
    except urllib.error.HTTPError as e:
        _log(f"FATAL: hub /ping HTTP {e.code} — {'token 不匹配' if e.code in (401, 403) else 'hub 异常'}")
        return -2
    except Exception as e:
        _log(f"FATAL: hub 不可达（{type(e).__name__}: {e}）——检查隧道 URL 是否过期")
        return -2

    try:
        code_raw = _fetch_code_zip(HUB_TOKEN, "hub")
    except Exception:
        return -2
    code_dir = _unpack_code(code_raw, Path("/tmp/worker-code"))
    sys.path.insert(0, str(code_dir))

    from remote.worker import supervise_worker

    # token 走 --token-file（H10：不进进程列表）；热替换 = 子进程退 86 → 监督器
    # 用同参重拉（fresh 进程加载新代码），kernel 与本 cell 的输出流不断。
    token_file = work_dir / "hub.token"
    token_file.write_text(HUB_TOKEN, encoding="utf-8")
    try:
        token_file.chmod(0o600)
    except Exception:
        pass
    max_idle = max(IDLE_FLOOR_SEC, (MAX_SESSION_HOURS - 1) * 3600)
    restart_argv = [
        "--poll", str(HUB_URL),
        "--token-file", str(token_file),
        "--out", str(work_dir),
        "--device", str(DEVICE),
        "--threads", "0",
        "--poll-sec", str(POLL_INTERVAL_SEC),
        "--max-idle-sec", str(max_idle),
    ]
    _log(f"worker 启动（max_idle={max_idle}s, poll={POLL_INTERVAL_SEC}s, device={DEVICE}）")
    try:
        return supervise_worker(restart_argv)
    except KeyboardInterrupt:
        _log("收到中断——worker 已终止")
        return 0


# ════════════════════════ 5. Push 模式 runner ════════════════════════
def run_push_worker() -> int:
    """启动 worker_server + cloudflared 隧道，等待 hub 推 job。返回码同 pull。"""
    import re

    work_dir = Path("/tmp/remote-worker-serve")
    work_dir.mkdir(parents=True, exist_ok=True)

    # ── cloudflared 查找 / 自动安装 ──
    cf_bin = CLOUDFLARED_PATH or subprocess.getoutput(
        "where cloudflared 2>nul || which cloudflared 2>/dev/null"
    ).strip()
    if not cf_bin:
        _log("cloudflared 未找到，自动安装…")
        try:
            cf_bin = "/usr/local/bin/cloudflared"
            subprocess.run(
                ["curl", "-fsSL",
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                 "-o", cf_bin],
                check=True, timeout=60,
            )
            os.chmod(cf_bin, 0o755)
            _log(f"cloudflared 已安装 -> {cf_bin}")
        except Exception as e:
            _log(f"cloudflared 自动安装失败: {e}——push 模式需要它暴露端口；或改用 pull 模式")
            return -2

    # ── bootstrap 代码：优先 hub /code（与 job 执行同源），GitHub tarball 兜底 ──
    # 语义边界：job 真正执行用的是 hub 随 job 下发的 code.zip；bootstrap 只负责把
    # 服务进程拉起来（它要能 import remote_worker_serve）。
    import importlib.util as _ilu

    boot_dir = Path("/tmp/push-bootstrap")
    if _ilu.find_spec("remote_worker_serve") is None:
        try:
            code_raw = _fetch_code_zip(PUSH_TOKEN, "hub")
            _unpack_code(code_raw, boot_dir)
        except Exception:
            url = "https://codeload.github.com/HuangJian/battle/tar.gz/refs/heads/main"
            _log(f"hub /code 不可用 —— GitHub tarball 兜底: {url}")
            try:
                import tarfile

                boot_dir.mkdir(parents=True, exist_ok=True)
                with urllib.request.urlopen(url, timeout=180) as r:
                    blob = r.read()
                with tarfile.open(fileobj=io.BytesIO(blob)) as tf:
                    tf.extractall(boot_dir)
                cands = sorted(boot_dir.glob("*/nn-training"))
                if cands:
                    boot_dir = cands[0]
                _log(f"bootstrap 代码就位 -> {boot_dir}")
            except Exception as e:
                _log(f"bootstrap 代码拉取失败: {e}——push 模式无法启动，建议改 pull 模式")
                return -2
    else:
        _log("remote_worker_serve 已可导入——跳过 bootstrap")

    # ── 启动 worker_server ──
    serve_log = work_dir / "serve.log"
    serve_env = dict(os.environ)
    serve_env["PYTHONPATH"] = (str(boot_dir) + os.pathsep + serve_env.get("PYTHONPATH", "")).rstrip(os.pathsep)
    with open(serve_log, "w") as log_f:
        serve_proc = subprocess.Popen(
            [sys.executable, "-u", "-m", "remote_worker_serve",
             "--port", str(PUSH_PORT), "--token", PUSH_TOKEN,
             "--work", str(work_dir), "--device", DEVICE],
            stdout=log_f, stderr=subprocess.STDOUT, env=serve_env,
        )
    _log(f"worker_server 启动 (PID {serve_proc.pid})，等就绪…")

    def _ping_ok() -> bool:
        try:
            req = urllib.request.Request(
                f"http://127.0.0.1:{PUSH_PORT}/ping",
                headers={"Authorization": f"Bearer {PUSH_TOKEN}"},
            )
            with urllib.request.urlopen(req, timeout=5) as r:
                return r.status == 200
        except Exception:
            return False

    t0 = time.time()
    while time.time() - t0 < 30 and not _ping_ok():
        time.sleep(1)
    if not _ping_ok():
        _log(f"worker_server 30s 未就绪——查 {serve_log}")
        serve_proc.kill()
        return -1
    _log("worker_server 就绪")

    # ── cloudflared 隧道 ──
    cf_proc = None
    cf_log = work_dir / "cloudflared.log"
    with open(cf_log, "w") as log_f:
        cf_proc = subprocess.Popen(
            [cf_bin, "tunnel", "--url", f"http://localhost:{PUSH_PORT}", "--logfile", str(cf_log)],
            stdout=log_f, stderr=subprocess.STDOUT,
        )
    cf_url = None
    t0 = time.time()
    while time.time() - t0 < 60:
        try:
            urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text(encoding="utf-8", errors="replace"))
            if urls:
                cf_url = urls[-1]
                break
        except Exception:
            pass
        if cf_proc.poll() is not None:
            _log(f"cloudflared 提前退出 (code {cf_proc.returncode})")
            break
        time.sleep(2)
    if cf_url:
        _log(f"★ 隧道 URL: {cf_url}")
        _log("把它配到 hub 侧 rl-config 对应节点的 url（gpu_push 节点），job 会自动推送至此")
    else:
        _log("⚠ 隧道 URL 未取得（超时/出错）——hub 推送将不可达；日志见 " + str(cf_log))

    # ── 等待 job（会话上限内守着） ──
    deadline = time.time() + MAX_SESSION_HOURS * 3600
    try:
        while True:
            if serve_proc.poll() is not None:
                _log(f"worker_server 退出 (code {serve_proc.returncode})")
                return serve_proc.returncode or 0
            if time.time() > deadline:
                _log(f"会话到顶 ({MAX_SESSION_HOURS}h)——干净收摊")
                return 0
            time.sleep(30)
    except KeyboardInterrupt:
        _log("收到中断")
        return 0
    finally:
        if cf_proc and cf_proc.poll() is None:
            cf_proc.kill()
            _log("cloudflared 已停")
        if serve_proc.poll() is None:
            serve_proc.kill()
            _log("worker_server 已停")


# ════════════════════════ 6. 主流程：banner → runner → 崩溃退避重启 ════════════════════════
KEEPALIVE_STOP.clear()
t_session = time.time()
_rc = 0
try:
    print("\n" + "=" * 62)
    print(f"  Battle GPU Worker | mode={MODE} platform={PLATFORM} device={DEVICE}")
    print(f"  会话上限 {MAX_SESSION_HOURS}h | 任务类型 ppo+bc | 停止 = 中断本 cell")
    print("=" * 62 + "\n")
    attempt = 0
    while True:
        if MODE == "pull":
            _rc = run_pull_worker()
        elif MODE == "push":
            _rc = run_push_worker()
        else:
            _log(f"未知 MODE={MODE!r}（可选 'pull' / 'push'）")
            _rc = -2
        if _rc == 0 or _rc == -2:
            break  # 干净退出 / 配置致命（token/URL 错——重启无意义）
        attempt += 1
        if attempt > MAX_WORKER_RESTARTS:
            _log(f"FATAL: worker 连续失败 {attempt} 次（rc={_rc}）——放弃自动重启")
            break
        backoff = min(30 * attempt, 300)
        _log(f"worker 异常退出 (rc={_rc})——{backoff}s 后第 {attempt}/{MAX_WORKER_RESTARTS} 次自动重启")
        time.sleep(backoff)
finally:
    KEEPALIVE_STOP.set()
    _elapsed = (time.time() - t_session) / 60
    print(f"\n[{time.strftime('%H:%M:%S')}] [battle-rl] 会话结束: rc={_rc}, 时长 {_elapsed:.1f} min")
